# C03 — Embeddings e Busca Vetorial

**Disciplina:** Sistemas Cognitivos com Large Language Models (INFNET, 26E2_3)

**Autor:** Anderson Corrêa

**Projeto:** Letra da Lei

Este notebook entrega o **ponto 3 da rubrica** (embeddings + busca vetorial).

O que será demonstrado:

1. Carregamento de um subconjunto do corpus
2. Chunking
3. Indexação vetorial com ChromaDB e consultas
4. Filtragem de metadados excluindo dispositivos `REVOGADO`.
5. Comparação entre `VectorIndex.query` e `hybrid_search` (BM25 + denso).
6. Gold-set, `hit_rate`/`MRR`, e um caso de falha discutido em detalhe.


In [1]:
import os

os.environ["HF_HUB_DISABLE_PROGRESS_BARS"] = "1"
os.environ["TRANSFORMERS_VERBOSITY"] = "error"
os.environ["TOKENIZERS_PARALLELISM"] = "false"

import sys
import time
from pathlib import Path

# make sure the local package is importable even if the kernel starts in a different cwd
_repo_root = Path.cwd()
if not (_repo_root / "direito_dados").exists():
    _repo_root = Path(__file__).resolve().parent if "__file__" in globals() else _repo_root
if str(_repo_root) not in sys.path:
    sys.path.insert(0, str(_repo_root))

from direito_dados.corpus import load_corpus, NORMS
from direito_dados.retrieval.chunks import chunk_corpus
from direito_dados.retrieval.embedder import E5Embedder
from direito_dados.retrieval.index import VectorIndex
from direito_dados.retrieval.lexical import BM25Index, hybrid_search
from direito_dados.retrieval.evaluate import GoldItem, evaluate


## 1. Subconjunto do corpus

O corpus completo tem 9 normas e 2.310 artigos (ver `c07_lei_como_dado.ipynb` para a
análise completa). Para este notebook, que gera embeddings reais com o
modelo `intfloat/multilingual-e5-base` (rodando em CPU), usamos um subconjunto **CP
(Código Penal) + L11343 (Lei de Drogas)** — suficiente para demonstrar a API de recuperação.


In [2]:
corpus = load_corpus("data/raw", specs=[NORMS["CP"], NORMS["L11343"]])
print(f"Normas carregadas: {[n.id for n in corpus.norms]}")
print(f"Total de artigos: {len(corpus.all_articles())}")
print(f"Em vigor: {len(corpus.in_force_articles())}")


Normas carregadas: ['CP', 'L11343']
Total de artigos: 545
Em vigor: 520


## 2. Como cada artigo é transformado em vetor

`chunk_corpus` gera um `Chunk` por artigo do corpus. O texto que efetivamente é embutido
(campo `embed_text`) não é o texto bruto do artigo: quando o artigo define um crime, o texto
embutido começa pela **rubrica** — o nome oficial do crime dado pelo legislador, como
"Furto", "Estelionato" ou "Homicídio simples" — seguida do ***caput***, a frase de abertura
do artigo que descreve a conduta central, e só então o texto completo do dispositivo
(truncado a 300 caracteres quando há rubrica; nem todo artigo define um crime novo, então
nem todo artigo tem uma).

Essa ordem é proposital: quando alguém busca "furto" ou "estelionato", é a rubrica que
carrega o sinal mais forte de que aquele artigo é a resposta certa, então ela lidera o texto
embutido; o *caput* vem logo em seguida para reforçar o sinal, porque muitos artigos têm
parágrafos e incisos longos que, sozinhos, diluiriam esses dois sinais. Vemos um exemplo
concreto na célula abaixo (`CP:art121`, cuja rubrica "Homicídio simples" abre o
`embed_text`).

O modelo de embeddings usado (`intfloat/multilingual-e5-base`, um modelo **e5**) exige que
o texto embutido seja marcado com um prefixo indicando seu papel: `"query: "` para
perguntas de busca e `"passage: "` para os textos indexados — é assim que o modelo foi
treinado (contrastive learning com pares query/passage). O `E5Embedder` aplica esses
prefixos automaticamente; omiti-los degrada a qualidade da recuperação.


In [3]:
chunks = chunk_corpus(corpus)
print(f"Chunks gerados (1 por artigo): {len(chunks)}")

example = next(c for c in chunks if c.id == "CP:art121")
print("\n--- Exemplo: CP:art121 (homicídio) ---")
print("id:        ", example.id)
print("citation:  ", example.metadata["citation"])
print("status:    ", example.metadata["status"])
print("text[:120]:", repr(example.text[:120]))
print("embed_text:", repr(example.embed_text))


Chunks gerados (1 por artigo): 524

--- Exemplo: CP:art121 (homicídio) ---
id:         CP:art121
citation:   CP art. 121
status:     alterado
text[:120]: 'Matar alguem:\nPena - reclusão, de seis a vinte anos.\nCaso de diminuição de pena\n§\n1º Se o agente comete o crime impelido'
embed_text: 'Homicídio simples. Matar alguem:. Matar alguem:\nPena - reclusão, de seis a vinte anos.\nCaso de diminuição de pena\n§\n1º Se o agente comete o crime impelido por motivo de relevante valor social ou moral, ou\nsob o domínio de violenta emoção, logo em seguida a injusta provocação da vítima, o\njuiz pode r'


### Higiene de dados: ids duplicados no snapshot

O parser deriva o número do artigo a partir do texto consolidado do Planalto. Em L11343,
alguns incisos com sufixo de letra do capítulo "Disposições Finais e Transitórias"
(ex.: Art. 8º-A a 8º-F, um item `(VETADO)`) colidem no mesmo id de chunk
(`L11343:art8`). O ChromaDB exige ids únicos, então deduplicamos aqui (mantendo a primeira ocorrência) antes
de indexar.

In [4]:
seen: set[str] = set()
unique_chunks = []
for c in chunks:
    if c.id in seen:
        continue
    seen.add(c.id)
    unique_chunks.append(c)

print(f"Chunks brutos:  {len(chunks)}")
print(f"Chunks únicos:  {len(unique_chunks)}  (descartados: {len(chunks) - len(unique_chunks)})")


Chunks brutos:  524
Chunks únicos:  524  (descartados: 0)


## 3. Índice vetorial (ChromaDB)

`VectorIndex.build` carrega o modelo e5 (download único na primeira execução), embute
todas as passagens com o prefixo `passage:` e monta uma coleção ChromaDB em memória.

In [5]:
embedder = E5Embedder()  # intfloat/multilingual-e5-base

t0 = time.time()
index = VectorIndex.build(unique_chunks, embedder)
print(f"Índice construído em {time.time() - t0:.1f}s ({len(unique_chunks)} passagens)")


Índice construído em 9.0s (524 passagens)


## 4. Consultas de exemplo

Consultas em linguagem natural. O embedder aplica o prefixo `query:`
internamente. Cada resultado traz `citation`, `score` (similaridade
de cosseno) e `metadata` (status de vigência, domínio, nível hierárquico).


In [6]:
def show_results(query, results):
    print(f'Consulta: "{query}"')
    for r in results:
        print(f"  {r.citation:<28} score={r.score:.3f}  status={r.metadata['status']}")
    print()

for q in [
    "qual a pena para quem mata alguém?",
    "furto de coisa alheia móvel",
    "tráfico ilícito de entorpecentes",
]:
    show_results(q, index.query(q, embedder, k=5))


Consulta: "qual a pena para quem mata alguém?"
  CP art. 121                  score=0.887  status=alterado
  CP art. 226                  score=0.860  status=alterado
  CP art. 212                  score=0.858  status=alterado
  CP art. 209                  score=0.855  status=vigente
  CP art. 258                  score=0.855  status=vigente

Consulta: "furto de coisa alheia móvel"
  CP art. 155                  score=0.890  status=alterado
  CP art. 168                  score=0.878  status=alterado
  CP art. 157                  score=0.877  status=alterado
  CP art. 156                  score=0.849  status=vigente
  CP art. 313                  score=0.841  status=alterado

Consulta: "tráfico ilícito de entorpecentes"
  L11343 art. 17               score=0.850  status=alterado
  L11343 art. 50-A             score=0.844  status=alterado
  L11343 art. 18               score=0.841  status=vigente
  L11343 art. 28               score=0.841  status=vigente
  CP art. 318                  

## 5. Vigência: filtragem por metadados

O comportamento **padrão** de `VectorIndex.query` é `exclude_revoked=True` — dispositivos
com `status == "revogado"` não aparecem em respostas a menos que sejam pedidos.

Para demonstração, usamos duas consultas direcionadas a artigos revogados do Código Penal:

- `"usurpação de nome ou pseudônimo alheio"` → CP art. 185 (revogado pela Lei nº 10.695/2003)
- `"violação sexual mediante fraude"` → CP art. 214 (revogado pela Lei nº 12.015/2009)


In [7]:
for q in ["usurpação de nome ou pseudônimo alheio", "violação sexual mediante fraude"]:
    print("=" * 70)
    print(f'Consulta: "{q}"')
    print("\n-- exclude_revoked=True (padrão) --")
    show_results(q, index.query(q, embedder, k=5))
    print("-- exclude_revoked=False --")
    show_results(q, index.query(q, embedder, k=5, exclude_revoked=False))


Consulta: "usurpação de nome ou pseudônimo alheio"

-- exclude_revoked=True (padrão) --
Consulta: "usurpação de nome ou pseudônimo alheio"
  CP art. 242                  score=0.866  status=alterado
  CP art. 162                  score=0.851  status=vigente
  CP art. 314                  score=0.850  status=vigente
  CP art. 313                  score=0.845  status=alterado
  CP art. 305                  score=0.844  status=vigente

-- exclude_revoked=False --
Consulta: "usurpação de nome ou pseudônimo alheio"
  CP art. 185                  score=0.914  status=revogado
  CP art. 242                  score=0.866  status=alterado
  CP art. 162                  score=0.851  status=vigente
  CP art. 314                  score=0.850  status=vigente
  CP art. 313                  score=0.845  status=alterado

Consulta: "violação sexual mediante fraude"

-- exclude_revoked=True (padrão) --
Consulta: "violação sexual mediante fraude"
  CP art. 215                  score=0.892  status=alterado


**Leitura do resultado acima:** os dois exemplos mostram por que o filtro de vigência
(`exclude_revoked`) importa — e por que, num dos dois casos, ele acaba sendo redundante.

Na primeira consulta (`"usurpação de nome ou pseudônimo alheio"`), com
`exclude_revoked=True` (padrão) `CP:art185` não aparece entre os 5 primeiros resultados.
Com `exclude_revoked=False`, o mesmo artigo passa a **liderar** o ranking (score 0.914, o
maior de todos) — porque a rubrica "Usurpação de nome ou pseudônimo alheio" está associada
a ele, mesmo o texto do dispositivo sendo hoje apenas a nota de revogação. Ou seja: um
artigo sem nenhum efeito jurídico consegue vencer a busca por similaridade só porque carrega
o nome do crime no texto embutido. É exatamente esse cenário que o filtro de vigência existe
para bloquear: sem ele, o resultado nº 1 seria uma lei revogada.

Na segunda consulta (`"violação sexual mediante fraude"`), o resultado é **idêntico** com o
filtro ligado ou desligado — mas por um motivo diferente. `CP:art214` (revogado pela Lei nº
12.015/2009) não carrega mais a rubrica desse crime: ela foi para `CP:art215`, o dispositivo
que sucedeu o art. 214 e hoje está em vigor (status `alterado`). Como `CP:art214` compete na
busca apenas com o texto da nota de revogação, ele nunca chega perto do topo — `CP:art215`
lidera com score 0.892 tanto com o filtro ligado quanto desligado. Aqui o filtro de vigência
é redundante para esta consulta específica (o revogado já perde por conta própria) — mas ele
continua sendo necessário em geral, como mostra o caso do `CP:art185` acima, em que o
revogado vencia sem o filtro.

O parâmetro `domain` funciona da mesma forma (filtro exato de metadado) — não exercitado
aqui porque este subconjunto tem um único domínio (`penal`); veja
`tests/retrieval/test_index.py` para o caso multi-domínio.


## 6. Denso vs. híbrido (BM25 + denso)

Este projeto usa três estratégias de recuperação, e vale definir cada uma antes de comparar:

- **Busca densa** (`VectorIndex.query`): compara o *significado* das frases via embeddings
  — funciona bem com sinônimos e paráfrases, porque não depende de as mesmas palavras
  aparecerem na pergunta e no texto.
- **BM25** (`BM25Index`): busca **léxica** clássica por sobreposição de palavras (TF-IDF/BM25,
  sem dependências externas) — funciona bem quando a pergunta usa exatamente as palavras do
  texto, mas não entende sinônimos.
- **Busca híbrida** (`hybrid_search`): combina as duas pontuações — normaliza os scores de
  cada retriever por min-max sobre seu próprio pool de candidatos e soma como
  `alpha * denso + (1 - alpha) * léxico` (`alpha=0.5` por padrão).

Comparamos as três estratégias porque os critérios de avaliação deste projeto pedem uma
comparação explícita de estratégias de recuperação — e porque, na prática, cada uma tem um
ponto forte diferente. As três consultas abaixo exploram esses pontos fortes e fracos.


In [8]:
bm25 = BM25Index.build(unique_chunks)

def compare(query, k=5):
    print("=" * 70)
    print(f'Consulta: "{query}"')
    print("  denso :", [r.id for r in index.query(query, embedder, k=k)])
    print("  bm25  :", [r.id for r in bm25.query(query, k=k)])
    print("  híbrido:", [r.id for r in hybrid_search(query, index, bm25, embedder, k=k)])

compare("tráfico ilícito de entorpecentes")
compare("estelionato mediante fraude")
compare("ficar com um dinheiro que recebeu emprestado e não devolver")


Consulta: "tráfico ilícito de entorpecentes"
  denso : ['L11343:art17', 'L11343:art50-A', 'L11343:art18', 'L11343:art28', 'CP:art318']
  bm25  : ['L11343:art73', 'L11343:art30', 'L11343:art17', 'L11343:art68', 'CP:art83']
  híbrido: ['L11343:art17', 'L11343:art73', 'L11343:art1', 'L11343:art30', 'L11343:art68']
Consulta: "estelionato mediante fraude"


  denso : ['CP:art171', 'CP:art170', 'CP:art215', 'CP:art206', 'CP:art183-A']
  bm25  : ['CP:art170', 'CP:art204', 'CP:art171', 'CP:art215', 'CP:art178']
  híbrido: ['CP:art171', 'CP:art170', 'CP:art215', 'CP:art204', 'CP:art178']
Consulta: "ficar com um dinheiro que recebeu emprestado e não devolver"
  denso : ['L11343:art62-A', 'CP:art349', 'L11343:art63-C', 'CP:art292', 'CP:art359-B']
  bm25  : ['CP:art313', 'CP:art357', 'CP:art292', 'CP:art168', 'CP:art356']


  híbrido: ['CP:art292', 'CP:art313', 'L11343:art62-A', 'CP:art349', 'L11343:art63-C']


**Leitura do resultado acima:** na primeira consulta (tráfico), os três retrievers
concordam em parte — sem nenhum caso extremo.

Na segunda consulta (`"estelionato mediante fraude"`), `CP:art171` (estelionato) lidera o
ranking **denso**, à frente de `CP:art170` e `CP:art215` — porque seu `embed_text` começa
com a rubrica ("Estelionato. Obter, para si..."), que casa diretamente com o termo da
consulta. O BM25 também recupera `CP:art171` (posição 3, por sobreposição lexical de termos
como "mediante" e variações de "fraude"), e o híbrido herda o mesmo topo do denso. Ou seja,
para esta consulta a fusão híbrida é redundante — o denso sozinho já resolve.

Isso levanta uma pergunta honesta: se o denso já encontra bem os nomes de crime, o híbrido
ainda tem alguma vantagem prática? Testamos uma terceira consulta desenhada
deliberadamente para **não** usar o vocabulário jurídico do dispositivo nem sua rubrica —
uma paráfrase coloquial da apropriação indébita (`CP:art168`, rubrica "Apropriação
indébita"): `"ficar com um dinheiro que recebeu emprestado e não devolver"`. O resultado:

- **Denso**: `CP:art168` não aparece entre os 5 primeiros — a paráfrase não compartilha
  vocabulário suficiente com "Apropriação indébita. Apropriar-se de coisa alheia móvel, de
  que tem a posse ou a detenção..." para que o embedding aproxime os dois.
- **BM25**: recupera `CP:art168` na 4ª posição — há sobreposição lexical parcial o
  suficiente com outros artigos do capítulo de crimes patrimoniais.
- **Híbrido**: mesmo assim, `CP:art168` **não aparece** no top-5 híbrido. A normalização
  min-max de `hybrid_search` combina `alpha * denso_norm + (1 - alpha) * léxico_norm`; como
  `CP:art168` está ausente do pool denso, seu `denso_norm` é 0.0, o que reduz pela metade
  (`alpha=0.5`) sua pontuação combinada e o deixa atrás de artigos que pontuam
  moderadamente nos dois retrievers ao mesmo tempo, mesmo sem ser a resposta certa para
  nenhum deles isoladamente.

Este é o caso de falha real desta seção: **a fusão híbrida não é uma rede de segurança
garantida**. Ela ajuda quando o léxico encontra algo que o denso perde por completo *e*
esse acerto lexical é forte o bastante para sobreviver à normalização conjunta. Mas quando a
consulta é uma paráfrase coloquial que não compartilha vocabulário nem com a rubrica nem com
o *caput* do dispositivo certo, denso e híbrido podem falhar juntos, e só o léxico puro,
isolado, ainda enxerga o sinal.


## 7. Avaliação de recuperação (gold-set)

Usamos um **gold-set**: uma lista pequena de perguntas cuja resposta certa (o id do
dispositivo correto) já é conhecida, para medir a qualidade da recuperação com números, em
vez de olhar só para exemplos isolados. Duas métricas, calculadas por `evaluate()`:

- **hit-rate@k**: a fração das perguntas cujo dispositivo correto aparece entre os *k*
  primeiros resultados.
- **MRR** (*mean reciprocal rank*, posição recíproca média): a média de `1/posição` do
  primeiro resultado correto de cada pergunta — quanto mais perto do topo o acerto aparece,
  mais perto de 1 fica o MRR (um acerto só na 5ª posição vale `1/5 = 0.2`, por exemplo).

O gold-set abaixo tem 6 perguntas. Cinco têm resposta dentro do subconjunto carregado (CP +
L11343); a sexta está **fora do corpus** (Código de Trânsito Brasileiro), incluída de
propósito para mostrar como o sistema se comporta diante de uma pergunta sem resposta
indexada.


In [9]:
gold = [
    GoldItem("Qual a pena para quem mata alguém?", ["CP:art121"]),
    GoldItem("Furto de coisa alheia móvel", ["CP:art155"]),
    GoldItem("Roubo mediante grave ameaça ou violência", ["CP:art157"]),
    GoldItem("Porte de drogas para consumo pessoal", ["L11343:art28"]),
    GoldItem("Associação de duas ou mais pessoas para o tráfico", ["L11343:art35"]),
    GoldItem("Homicídio culposo na direção de veículo automotor", ["CTB:art302"]),  # out of corpus (on purpose)
]

def dense_retriever(query, k):
    return index.query(query, embedder, k=k)

metrics = evaluate(dense_retriever, gold, k=5)
print("Métricas (retriever denso, k=5):", metrics)

print()
for item in gold:
    results = dense_retriever(item.question, 5)
    ids = [r.id for r in results]
    is_hit = any(rid in item.relevant_ids for rid in ids)
    label = "ACERTO" if is_hit else "FALHA "
    print(f"[{label}] {item.question}")
    print(f"         esperado={item.relevant_ids}  top-5={ids}")


Métricas (retriever denso, k=5): {'hit_rate': 0.8333333333333334, 'mrr': 0.8333333333333334, 'n': 6}



[ACERTO] Qual a pena para quem mata alguém?
         esperado=['CP:art121']  top-5=['CP:art121', 'CP:art258', 'CP:art226', 'CP:art212', 'CP:art287']
[ACERTO] Furto de coisa alheia móvel
         esperado=['CP:art155']  top-5=['CP:art155', 'CP:art157', 'CP:art168', 'CP:art156', 'CP:art312']
[ACERTO] Roubo mediante grave ameaça ou violência
         esperado=['CP:art157']  top-5=['CP:art157', 'CP:art359-L', 'CP:art359-J', 'CP:art359-M', 'CP:art197']


[ACERTO] Porte de drogas para consumo pessoal
         esperado=['L11343:art28']  top-5=['L11343:art28', 'CP:art270', 'L11343:art66', 'L11343:art26', 'L11343:art33']
[ACERTO] Associação de duas ou mais pessoas para o tráfico
         esperado=['L11343:art35']  top-5=['L11343:art35', 'CP:art288', 'CP:art149-A', 'L11343:art37', 'L11343:art6']
[FALHA ] Homicídio culposo na direção de veículo automotor
         esperado=['CTB:art302']  top-5=['CP:art73', 'CP:art74', 'CP:art359-M-A', 'L11343:art62', 'CP:art234-C']


### Discussão do caso de falha

A pergunta sobre homicídio culposo na direção de veículo, cuja resposta correta
(`CTB:art302`) está no Código de Trânsito Brasileiro, falha como esperado: o corpus deste
projeto cobre o microssistema penal federal (CF, CP, CPP, LEP e as leis extravagantes
penais listadas em `NORMS`), e o CTB não faz parte dele. Como `VectorIndex.query` sempre
devolve os *k* vizinhos mais próximos, a consulta retorna os artigos do CP/L11343
semanticamente mais próximos (sobre culpa, trânsito, condução) mesmo sem nenhum deles ser a
resposta certa. Isso evidencia um limite real do design: recuperação por *k* fixo não
distingue "não sei" de "a melhor opção que eu tenho" — quem consome esse resultado (a
camada de geração, em `direito_dados.generation.rag`) precisa de um mecanismo à parte, de
verificação de citação e abstenção, para não tratar um resultado ruim como se fosse bom.

O `hit_rate` obtido (5/6 ≈ 0,83) reflete as cinco perguntas plenamente respondidas pelo
subconjunto carregado. O `MRR` é igual ao `hit_rate` — o que só acontece quando toda
pergunta respondida corretamente acerta o dispositivo certo já na **1ª posição** do
ranking, não em algum lugar dentro do top-5 (compare com o `top-5` de cada item `ACERTO`
acima: o id esperado é sempre o primeiro da lista). Isso acontece porque a rubrica de cada
crime (seu nome oficial) lidera o texto embutido de cada artigo, o que faz a busca por nome
de crime (ex.: "Furto de coisa alheia móvel") apontar diretamente para o artigo certo com
folga de score sobre concorrentes próximos (ex.: `CP:art155` vs. `CP:art168`, ver seção 4).


## Conclusão

Este notebook demonstrou o pipeline completo de recuperação: chunking rubrica- e
*caput*-forward → embeddings e5 com prefixos assimétricos → índice ChromaDB com filtragem
de metadados → comparação denso/BM25/híbrido → avaliação com gold-set.

O ponto mais importante para um sistema de "direito como dado" é o de segurança de
vigência (seção 5): a exclusão de dispositivos revogados por padrão não é um efeito
colateral do ranking semântico, é uma condição de filtro explícita aplicada antes da
consulta vetorial — a garantia central de que este RAG nunca cita, por engano, uma lei que
não vale mais.

Liderar o texto embutido de cada artigo com sua rubrica (seções 2, 5 e 6) faz consultas por
nome de crime serem resolvidas corretamente pela busca densa sozinha, com folga de score e
na 1ª posição — é por isso que o `MRR` do gold-set (seção 7) coincide com o `hit_rate`. Mas
a seção 6 também deixou claro que a fusão híbrida não é uma rede de segurança universal:
para uma paráfrase coloquial que não compartilha vocabulário nem com a rubrica nem com o
*caput* do dispositivo certo, denso e híbrido podem falhar juntos, e só o léxico puro,
isolado, ainda enxerga o sinal. Isso reforça, junto com a falha estrutural do item fora de
escopo (seção 7), que recuperação por similaridade — com ou sem fusão — não substitui um
mecanismo explícito de verificação e abstenção na camada de geração.
